In [3]:
"""Bedrock client wrappers for embeddings and LLM calls."""
import json
import logging
from functools import lru_cache
import boto3
from src.rag.config import settings


log = logging.getLogger(__name__)
# Add at the top with other constants
HAIKU = "us.anthropic.claude-3-5-haiku-20241022-v1:0"
SONNET = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"


def claude_invoke(
    prompt: str,
    system: str = "",
    model_id: str = HAIKU,
    max_tokens: int = 1024,
    temperature: float = 0.0,
) -> str:
    """
    Call Claude via Bedrock with a single user message.
    
    Args:
        prompt: The user message
        system: System prompt (instructions about role/behavior)
        model_id: HAIKU (cheap/fast) or SONNET (smart/expensive)
        max_tokens: Cap on response length
        temperature: 0 = deterministic, 1 = creative. RAG wants 0.
    
    Returns:
        The text response from Claude.
    """
    client = get_bedrock_runtime()
    
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "system": system,
        "messages": [{"role": "user", "content": prompt}],
    })
    
    response = client.invoke_model(
        modelId=model_id,
        body=body,
        contentType="application/json",
        accept="application/json",
    )
    
    result = json.loads(response["body"].read())
    # Anthropic responses have a content array with text blocks
    return result["content"][0]["text"]

@lru_cache(maxsize=1)
def get_bedrock_runtime():
    """
    Cached Bedrock runtime client.
    
    @lru_cache ensures we create ONE client per process — boto3 clients
    are thread-safe and expensive to construct. Recreating per call
    would add ~100ms latency.
    """
    session = boto3.Session(profile_name=settings.aws_profile)
    return session.client("bedrock-runtime", region_name=settings.aws_region)


# Titan Text Embeddings V2 model ID
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
EMBEDDING_DIM = 1024


def embed_text(text: str, dimensions: int = EMBEDDING_DIM) -> list[float]:
    """
    Embed a single piece of text using Titan v2.
    
    Args:
        text: Input text (max ~8000 tokens / ~30k chars)
        dimensions: 1024 (default), 512, or 256. Lower = cheaper but less precise.
    
    Returns:
        List of `dimensions` floats.
    """
    if not text.strip():
        raise ValueError("Cannot embed empty text")
    
    client = get_bedrock_runtime()
    
    body = json.dumps({
        "inputText": text,
        "dimensions": dimensions,
        "normalize": True,  # Returns unit vectors; faster cosine search
    })
    
    response = client.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=body,
        contentType="application/json",
        accept="application/json",
    )
    
    result = json.loads(response["body"].read())
    return result["embedding"]


def embed_batch(texts: list[str], dimensions: int = EMBEDDING_DIM) -> list[list[float]]:
    """
    Embed a list of texts. Titan v2 doesn't support native batching via Bedrock,
    so we loop (still fast since each call is ~50ms).
    
    For real production with high volume, you'd use Bedrock batch inference jobs.
    """
    return [embed_text(t, dimensions) for t in texts]

In [5]:
body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1024,
        "temperature": 0.3,
        "system": "",
        "messages": [{"role": "user", "content": "what is AI?"}],
    })

In [6]:
client = get_bedrock_runtime()

In [24]:
response = client.invoke_model(
        modelId="minimax.minimax-m2.5",
        body=body,
        contentType="application/json",
        accept="application/json",
    )

In [14]:
response

{'ResponseMetadata': {'RequestId': '7d269de0-7db7-419b-a330-079097ba1d4d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 13 May 2026 19:35:05 GMT',
   'content-type': 'application/json',
   'content-length': '4338',
   'connection': 'keep-alive',
   'x-amzn-requestid': '7d269de0-7db7-419b-a330-079097ba1d4d',
   'x-amzn-bedrock-invocation-latency': '2782',
   'x-amzn-bedrock-output-token-count': '823',
   'x-amzn-bedrock-input-token-count': '71'},
  'RetryAttempts': 0},
 'contentType': 'application/json',
 'body': <botocore.response.StreamingBody at 0x10bb90520>}

In [18]:
!aws bedrock list-inference-profiles

{
    "inferenceProfileSummaries": [
        {
            "inferenceProfileName": "US Anthropic Claude 3 Sonnet",
            "description": "Routes requests to Anthropic Claude 3 Sonnet in us-east-1 and us-west-2.",
            "createdAt": "2024-08-26T00:00:00+00:00",
            "updatedAt": "2024-08-26T00:00:00+00:00",
            "inferenceProfileArn": "arn:aws:bedrock:us-east-1:665908544460:inference-profile/us.anthropic.claude-3-sonnet-20240229-v1:0",
            "models": [
                {
                    "modelArn": "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0"
                },
                {
                    "modelArn": "arn:aws:bedrock:us-west-2::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0"
                }
            ],
            "inferenceProfileId": "us.anthropic.claude-3-sonnet-20240229-v1:0",
            "status": "ACTIVE",
            "type": "SYSTEM_DEFINED"
        },
        {
            "infer